# ⬡ DriveSense AI — Google Colab Runner
**ITU Lahore · BSAI · Software Engineering Final Project**  
Qasim Bin Shahzad (BSAI-24070) · Salman Ejaz Jathol (BSAI-24067)

---
### Instructions
1. Make sure Runtime → Change runtime type → **T4 GPU** is selected
2. Run all cells top to bottom (**Runtime → Run all**)
3. When the Gradio link appears, click it to open the app
4. Upload your dashcam footage and click **ANALYSE FOOTAGE**

> **Note:** The public Gradio URL is valid for 72 hours. Save it as your submission URL.

In [ ]:
# ── Cell 1: Install system dependencies ──────────────────────────────────────
!apt-get update -qq
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ ffmpeg installed')

In [ ]:
# ── Cell 2: Install Python packages ──────────────────────────────────────────
!pip install -q ultralytics gradio opencv-python-headless numpy
print('✅ Python packages installed')

In [ ]:
# ── Cell 3: Write detector.py ─────────────────────────────────────────────────
detector_code = '''
"""
detector.py  –  DriveSense AI  |  Core Detection Engine
"""

import cv2
import numpy as np
from ultralytics import YOLO
from dataclasses import dataclass
from typing import Optional

COCO_CAR_IDS    = {2, 5, 7}
COCO_LIGHT_ID   = 9
FOCAL_CONST     = 800
VEHICLE_WIDTH_M = 1.9
OWN_CAR_MASK_FRAC_Y  = 0.82
OWN_CAR_MASK_FRAC_X  = (0.25, 0.75)
LIGHT_MIN_CONF = 0.40


@dataclass
class DriverProfile:
    vehicle_name    : str   = "Standard Car"
    braking_100_sec : float = 3.5
    speed_limit_kmh : float = 70.0

    @property
    def safe_distance_m(self) -> float:
        v = self.speed_limit_kmh / 3.6
        t_reaction = 1.5
        v100 = 100 / 3.6
        a = v100 / max(self.braking_100_sec, 0.5)
        d_braking = (v ** 2) / (2 * a)
        d_reaction = v * t_reaction
        return (d_reaction + d_braking) * 1.2


@dataclass
class FrameEvent:
    frame_idx          : int
    timestamp_s        : float
    nearest_car_dist_m : Optional[float] = None
    too_close          : bool = False
    lane_status        : str  = "OK"
    light_detected     : bool = False
    light_color        : str  = "NONE"
    light_violation    : bool = False
    annotated_frame    : Optional[object] = None


class DriveSenseDetector:

    def __init__(self, profile: DriverProfile, model_name: str = "yolo11n.pt"):
        print(f"[DriveSense] Loading model: {model_name}")
        self.model   = YOLO(model_name)
        self.profile = profile
        self._green_stationary_frames = 0
        self._prev_gray = None

    def analyse_frame(self, frame, frame_idx, fps, green_frames_threshold):
        h, w = frame.shape[:2]
        ts   = frame_idx / max(fps, 1)
        event = FrameEvent(frame_idx=frame_idx, timestamp_s=ts)

        results = self.model(frame, verbose=False, conf=0.35)
        boxes   = results[0].boxes if results else []

        car_boxes, light_boxes = [], []
        for box in boxes:
            cls_id = int(box.cls[0].item())
            conf   = float(box.conf[0].item())
            xyxy   = box.xyxy[0].cpu().numpy().astype(int)
            if cls_id in COCO_CAR_IDS:
                if not self._is_own_car(xyxy, h, w):
                    car_boxes.append((xyxy, conf))
            elif cls_id == COCO_LIGHT_ID and conf >= LIGHT_MIN_CONF:
                light_boxes.append((xyxy, conf))

        nearest_dist, nearest_box = None, None
        for xyxy, conf in car_boxes:
            dist = self._estimate_distance(xyxy)
            if dist is not None and (nearest_dist is None or dist < nearest_dist):
                nearest_dist, nearest_box = dist, xyxy

        event.nearest_car_dist_m = nearest_dist
        if nearest_dist is not None:
            event.too_close = nearest_dist < self.profile.safe_distance_m

        if light_boxes:
            event.light_detected = True
            best_xyxy = max(light_boxes, key=lambda x: x[1])[0]
            event.light_color = self._classify_light_color(frame, best_xyxy)

        is_moving = self._is_car_moving(frame)
        if event.light_color == "GREEN" and not is_moving and nearest_dist is None:
            self._green_stationary_frames += 1
            if self._green_stationary_frames >= green_frames_threshold:
                event.light_violation = True
        else:
            self._green_stationary_frames = 0

        event.lane_status = self._check_lane(frame)
        event.annotated_frame = self._annotate(
            frame.copy(), event, car_boxes, light_boxes, nearest_box, nearest_dist, h, w
        )
        self._prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return event

    def _is_own_car(self, xyxy, h, w):
        x1, y1, x2, y2 = xyxy
        bottom_frac = y2 / h
        cx_frac     = ((x1 + x2) / 2) / w
        return (bottom_frac > OWN_CAR_MASK_FRAC_Y and
                OWN_CAR_MASK_FRAC_X[0] < cx_frac < OWN_CAR_MASK_FRAC_X[1])

    def _estimate_distance(self, xyxy):
        x1, y1, x2, y2 = xyxy
        apparent_w = x2 - x1
        if apparent_w < 5:
            return None
        return round(FOCAL_CONST / apparent_w, 1)

    def _classify_light_color(self, frame, xyxy):
        x1, y1, x2, y2 = xyxy
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            return "UNKNOWN"
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        h_c = crop.shape[0]
        thirds = [hsv[:h_c//3], hsv[h_c//3:2*h_c//3], hsv[2*h_c//3:]]
        names = ["RED", "YELLOW", "GREEN"]
        hue_ranges = [[(0,15),(160,180)], [(20,38)], [(38,90)]]
        scores = []
        for third, ranges in zip(thirds, hue_ranges):
            mask = np.zeros(third.shape[:2], dtype=np.uint8)
            for lo, hi in ranges:
                mask |= cv2.inRange(third, (lo,60,60), (hi,255,255))
            scores.append(int(mask.sum()))
        best = int(np.argmax(scores))
        return names[best] if scores[best] >= 50 else "UNKNOWN"

    def _is_car_moving(self, frame):
        if self._prev_gray is None:
            return True
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        roi_prev = self._prev_gray[h//4:3*h//4, w//4:3*w//4]
        roi_curr = gray[h//4:3*h//4, w//4:3*w//4]
        diff = cv2.absdiff(roi_prev, roi_curr)
        return float(diff.mean()) > 1.5

    def _check_lane(self, frame):
        h, w = frame.shape[:2]
        roi_vertices = np.array([[
            (int(w*0.0), h), (int(w*0.45), int(h*0.55)),
            (int(w*0.55), int(h*0.55)), (int(w*1.0), h),
        ]], dtype=np.int32)
        gray   = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur   = cv2.GaussianBlur(gray, (7,7), 0)
        edges  = cv2.Canny(blur, 40, 120)
        mask   = np.zeros_like(edges)
        cv2.fillPoly(mask, roi_vertices, 255)
        masked = cv2.bitwise_and(edges, mask)
        lines  = cv2.HoughLinesP(masked, 1, np.pi/180, 40, minLineLength=50, maxLineGap=80)
        if lines is None:
            return "OK"
        left_xs, right_xs = [], []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 == x1:
                continue
            slope = (y2 - y1) / (x2 - x1)
            if abs(slope) < 0.3:
                continue
            x_bottom = int(x1 + (h - y1) / slope) if abs(slope) > 0.001 else x1
            (left_xs if slope < 0 else right_xs).append(x_bottom)
        left_x  = int(np.mean(left_xs))  if left_xs  else None
        right_x = int(np.mean(right_xs)) if right_xs else None
        centre  = w // 2
        OVER_THRESH = int(w * 0.08)
        if left_x  is not None and abs(left_x  - centre) < OVER_THRESH: return "OVER_LINE"
        if right_x is not None and abs(right_x - centre) < OVER_THRESH: return "OVER_LINE"
        if left_xs  and not right_xs: return "DRIFT"
        if right_xs and not left_xs:  return "DRIFT"
        return "OK"

    def _annotate(self, frame, event, car_boxes, light_boxes, nearest_box, nearest_dist, h, w):
        for xyxy, conf in car_boxes:
            dist       = self._estimate_distance(xyxy)
            is_nearest = nearest_box is not None and np.array_equal(xyxy, nearest_box)
            color      = (0,0,220) if (is_nearest and event.too_close) else (50,205,50)
            label      = f"{dist:.0f}m" if dist else "car"
            if is_nearest and event.too_close: label += " WARNING: TOO CLOSE"
            cv2.rectangle(frame, (xyxy[0],xyxy[1]), (xyxy[2],xyxy[3]), color, 2)
            self._put_label(frame, label, (xyxy[0], xyxy[1]-8), color)
        for xyxy, conf in light_boxes:
            color_map = {"RED":(0,0,255),"GREEN":(0,220,0),"YELLOW":(0,210,255),"UNKNOWN":(200,200,200)}
            c = color_map.get(event.light_color, (200,200,200))
            cv2.rectangle(frame, (xyxy[0],xyxy[1]), (xyxy[2],xyxy[3]), c, 2)
            self._put_label(frame, f"LIGHT:{event.light_color}", (xyxy[0],xyxy[1]-8), c)
        panel_h = 95
        overlay = frame.copy()
        cv2.rectangle(overlay, (0,0), (w, panel_h), (15,15,25), -1)
        cv2.addWeighted(overlay, 0.72, frame, 0.28, 0, frame)
        if nearest_dist is not None:
            safe_d   = self.profile.safe_distance_m
            dist_col = (0,80,255) if event.too_close else (60,220,60)
            cv2.putText(frame, f"DIST: {nearest_dist:.0f}m  |  SAFE: {safe_d:.0f}m",
                        (12,28), cv2.FONT_HERSHEY_DUPLEX, 0.65, dist_col, 1, cv2.LINE_AA)
        else:
            cv2.putText(frame, "DIST: --", (12,28), cv2.FONT_HERSHEY_DUPLEX, 0.65, (180,180,180), 1, cv2.LINE_AA)
        lane_col = {"OK":(60,220,60),"DRIFT":(0,180,255),"OVER_LINE":(0,60,255)}.get(event.lane_status,(180,180,180))
        cv2.putText(frame, f"LANE: {event.lane_status}", (12,58), cv2.FONT_HERSHEY_DUPLEX, 0.65, lane_col, 1, cv2.LINE_AA)
        if event.light_detected:
            txt = f"LIGHT: {event.light_color}" + ("  !! FAILED TO MOVE !!" if event.light_violation else "")
            lc  = (0,0,255) if event.light_violation else (200,200,200)
            cv2.putText(frame, txt, (12,88), cv2.FONT_HERSHEY_DUPLEX, 0.65, lc, 1, cv2.LINE_AA)
        cv2.putText(frame, f"{event.timestamp_s:.1f}s", (w-100,28), cv2.FONT_HERSHEY_DUPLEX, 0.6, (160,160,160), 1, cv2.LINE_AA)
        return frame

    @staticmethod
    def _put_label(frame, text, pos, color, scale=0.5, thick=1):
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thick)
        x, y = pos
        cv2.rectangle(frame, (x-2,y-th-4), (x+tw+2,y+2), (10,10,10), -1)
        cv2.putText(frame, text, (x,y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick, cv2.LINE_AA)
'''
with open('detector.py', 'w') as f:
    f.write(detector_code.strip())
print('✅ detector.py written')

In [ ]:
# ── Cell 4: Write scorer.py ───────────────────────────────────────────────────
scorer_code = '''
from __future__ import annotations
from typing import List
from dataclasses import dataclass
from detector import FrameEvent, DriverProfile

WEIGHTS   = {"following_distance":0.40, "lane_discipline":0.35, "traffic_light":0.25}
PENALTIES = {"too_close_per_frame":0.15, "lane_drift_per_event":0.8,
             "lane_over_per_event":1.2, "light_violation":2.5}

@dataclass
class SafetyReport:
    overall_score:float; distance_score:float; lane_score:float; light_score:float
    total_frames:int; analysed_frames:int; duration_s:float
    too_close_count:int; drift_events:int; over_line_events:int
    light_violations:int; lights_seen:int; vehicle_name:str
    suggestions:List[str]; markdown:str

class SafetyScorer:
    def __init__(self, profile:DriverProfile):
        self.profile = profile

    def score(self, events:List[FrameEvent]) -> SafetyReport:
        if not events: return self._empty_report()
        n = len(events)
        duration_s = events[-1].timestamp_s
        too_close_frames = sum(1 for e in events if e.too_close)
        car_visible = sum(1 for e in events if e.nearest_car_dist_m is not None)
        dist_penalty = min(10.0,(too_close_frames/car_visible)*18.0) if car_visible>0 else 0.0
        distance_score = max(0.0, 10.0 - dist_penalty)
        drift_events, over_line_events = self._count_lane_events(events)
        lane_score = max(0.0, 10.0 - drift_events*0.8 - over_line_events*1.2)
        light_violations = sum(1 for e in events if e.light_violation)
        lights_seen = sum(1 for e in events if e.light_detected)
        light_score = max(0.0, 10.0 - light_violations*2.5)
        overall = round(min(10.0,max(0.0,
            WEIGHTS["following_distance"]*distance_score +
            WEIGHTS["lane_discipline"]*lane_score +
            WEIGHTS["traffic_light"]*light_score)),1)
        suggestions = self._build_suggestions(
            distance_score,lane_score,light_score,
            too_close_frames,drift_events,over_line_events,light_violations,car_visible)
        md = self._build_markdown(overall,distance_score,lane_score,light_score,
            n,duration_s,too_close_frames,drift_events,over_line_events,
            light_violations,lights_seen,suggestions)
        return SafetyReport(overall,round(distance_score,1),round(lane_score,1),
            round(light_score,1),n,n,round(duration_s,1),too_close_frames,
            drift_events,over_line_events,light_violations,lights_seen,
            self.profile.vehicle_name,suggestions,md)

    @staticmethod
    def _count_lane_events(events):
        d=o=0; prev="OK"
        for e in events:
            if e.lane_status=="DRIFT" and prev!="DRIFT": d+=1
            elif e.lane_status=="OVER_LINE" and prev!="OVER_LINE": o+=1
            prev=e.lane_status
        return d,o

    def _build_suggestions(self,dist_sc,lane_sc,light_sc,too_close,drifts,over_lines,light_viols,car_visible):
        tips=[]
        if dist_sc<7.0:
            tips.append(f"⚠️ **Following Distance**: Too close in {too_close} frames. "
                f"At {self.profile.speed_limit_kmh:.0f} km/h, maintain at least **{self.profile.safe_distance_m:.0f} m**. "
                f"Your {self.profile.vehicle_name} needs ~{self.profile.braking_100_sec:.1f}s to stop from 100 km/h.")
        elif dist_sc<9.0:
            tips.append(f"✅ Following distance mostly good. Keep the {self.profile.safe_distance_m:.0f} m buffer as a habit.")
        if over_lines>0:
            tips.append(f"⚠️ **Over the Line**: Straddled lane markings {over_lines} time(s). Keep tyres well within boundaries.")
        if drifts>0:
            tips.append(f"⚠️ **Lane Drift**: {drifts} drift event(s). Check mirrors frequently and keep both hands on the wheel.")
        if lane_sc>=9.5: tips.append("✅ Excellent lane discipline throughout the clip.")
        if light_viols>0:
            tips.append(f"⚠️ **Traffic Light**: Failed to move {light_viols} time(s) on green. Stay alert at junctions.")
        elif light_sc==10.0: tips.append("✅ Good signal awareness — no green-light hesitation.")
        if not tips: tips.append("🏆 Outstanding drive! No significant issues detected.")
        return tips

    def _build_markdown(self,overall,dist_sc,lane_sc,light_sc,frames,duration,
                         too_close,drifts,over_lines,light_viols,lights_seen,suggestions):
        def stars(s): f=round(s/2); return "★"*f+"☆"*(5-f)
        grade=next(g for t,g in [(9,"A+"),(8,"A"),(7,"B"),(6,"C"),(5,"D"),(0,"F")] if overall>=t)
        lines=[
            f"# 🚗 DriveSense AI — Safety Report","",
            f"**Vehicle:** {self.profile.vehicle_name}  |  **Clip:** {duration:.1f}s  |  **Frames:** {frames}","","---","",
            f"## Overall Score:  {overall} / 10   ({grade})","",
            f"| Category | Score | Stars |",f"|----------|-------|-------|",
            f"| 🚘 Following Distance | {dist_sc:.1f}/10 | {stars(dist_sc)} |",
            f"| 🛣️ Lane Discipline    | {lane_sc:.1f}/10 | {stars(lane_sc)} |",
            f"| 🚦 Traffic Lights     | {light_sc:.1f}/10 | {stars(light_sc)} |","","---","",
            f"## Incident Summary","",
            f"- Too-close frames: {too_close}",f"- Lane drifts: {drifts}",
            f"- Over-line events: {over_lines}",f"- Light violations: {light_viols}",
            f"- Lights detected: {lights_seen}","","---","",f"## Feedback",""
        ]
        for tip in suggestions: lines.append(f"- {tip}")
        lines+=["","---","*Generated by DriveSense AI*"]
        return "\n".join(lines)

    def _empty_report(self):
        return SafetyReport(0.0,0.0,0.0,0.0,0,0,0.0,0,0,0,0,0,
            self.profile.vehicle_name,["No frames analysed."],"# Error\n\nNo frames analysed.")
'''
with open('scorer.py', 'w') as f:
    f.write(scorer_code.strip())
print('✅ scorer.py written')

In [ ]:
# ── Cell 5: Write app.py ──────────────────────────────────────────────────────
app_code = '''
import cv2, gradio as gr, numpy as np, os, subprocess, tempfile
from typing import Optional
from detector import DriveSenseDetector, DriverProfile
from scorer   import SafetyScorer

DESIRED_FPS        = 12
MODEL_NAME         = "yolo11n.pt"
GREEN_STATIONARY_S = 2.0

def _reencode_for_browser(raw_path):
    out_path = raw_path.replace(".mp4", "_h264.mp4")
    try:
        result = subprocess.run(
            ["ffmpeg","-y","-i",raw_path,"-vcodec","libx264","-pix_fmt","yuv420p",
             "-preset","fast","-crf","23","-movflags","+faststart",out_path],
            capture_output=True, timeout=300)
        if result.returncode==0 and os.path.exists(out_path):
            os.remove(raw_path)
            return out_path
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return raw_path

def process_video(input_video, vehicle_name, braking_time_s, speed_limit, progress=gr.Progress(track_tqdm=False)):
    if input_video is None:
        return None, "Please upload a video.", ""
    profile  = DriverProfile(vehicle_name=vehicle_name, braking_100_sec=braking_time_s, speed_limit_kmh=speed_limit)
    detector = DriveSenseDetector(profile=profile, model_name=MODEL_NAME)
    scorer   = SafetyScorer(profile=profile)
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        return None, "Could not open video.", ""
    orig_fps     = max(1.0, cap.get(cv2.CAP_PROP_FPS) or 25)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    interval     = max(1, round(orig_fps / DESIRED_FPS))
    green_thresh = max(1, round(GREEN_STATIONARY_S * DESIRED_FPS))
    out_path = os.path.join(tempfile.gettempdir(), "drivesense_output.mp4")
    out      = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), DESIRED_FPS, (width,height))
    events, frame_idx = [], 0
    progress(0, desc="Initialising…")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_idx % interval == 0:
            event = detector.analyse_frame(frame, frame_idx, fps=orig_fps, green_frames_threshold=green_thresh)
            events.append(event)
            out.write(event.annotated_frame if event.annotated_frame is not None else frame)
            progress(min(0.99, frame_idx/max(total_frames,1)), desc=f"Frame {frame_idx}/{total_frames}")
        frame_idx += 1
    cap.release(); out.release()
    progress(1.0, desc="Generating report…")
    out_path = _reencode_for_browser(out_path)
    report   = scorer.score(events)
    return out_path, report.markdown, _build_score_html(report)

def _score_color(s):
    return "#22c55e" if s>=8 else "#f59e0b" if s>=6 else "#ef4444"

def _build_score_html(report):
    def gauge(label, score):
        pct=score*10; color=_score_color(score)
        return f"""<div style=\'margin:10px 0;\'><div style=\'display:flex;justify-content:space-between;margin-bottom:4px;\'><span style=\'font-family:monospace;font-size:13px;color:#ccc;\'>{label}</span><span style=\'font-family:monospace;font-size:13px;color:{color};font-weight:bold;\'>{score:.1f}/10</span></div><div style=\'background:#1e1e2e;border-radius:6px;height:10px;overflow:hidden;\'><div style=\'width:{pct}%;height:100%;background:{color};border-radius:6px;\'></div></div></div>"""
    oc=_score_color(report.overall_score)
    grade=next(g for t,g in [(9,"A+"),(8,"A"),(7,"B"),(6,"C"),(5,"D"),(0,"F")] if report.overall_score>=t)
    return f"""<div style=\'background:#0f0f1a;border:1px solid #2a2a3e;border-radius:12px;padding:22px 28px;font-family:monospace;color:#e0e0e0;\'><div style=\'text-align:center;margin-bottom:20px;\'><div style=\'font-size:11px;letter-spacing:3px;color:#666;\'>SAFETY SCORE</div><div style=\'font-size:56px;font-weight:900;color:{oc};line-height:1;\'>{report.overall_score}</div><div style=\'font-size:18px;color:{oc};\'>/ 10 ({grade})</div><div style=\'font-size:11px;color:#555;\'>{report.duration_s:.1f}s · {report.analysed_frames} frames · {report.vehicle_name}</div></div><div style=\'border-top:1px solid #2a2a3e;padding-top:16px;\'>{gauge("🚘 Following Distance",report.distance_score)}{gauge("🛣️ Lane Discipline",report.lane_score)}{gauge("🚦 Traffic Lights",report.light_score)}</div><div style=\'border-top:1px solid #2a2a3e;padding-top:14px;margin-top:6px;font-size:12px;color:#888;display:grid;grid-template-columns:1fr 1fr;gap:6px;\'><div>Too-close: <b style=\'color:#ccc;\'>{report.too_close_count}</b></div><div>Lane drifts: <b style=\'color:#ccc;\'>{report.drift_events}</b></div><div>Over-line: <b style=\'color:#ccc;\'>{report.over_line_events}</b></div><div>Light violations: <b style=\'color:#ccc;\'>{report.light_violations}</b></div></div></div>"""

VEHICLE_PRESETS = {
    "Suzuki Mehran (650cc)":("Suzuki Mehran",5.2),
    "Suzuki Alto / Wagon R":("Suzuki Alto",4.5),
    "Toyota Corolla / Civic":("Toyota Corolla",3.5),
    "Honda City / Yaris":("Honda City",3.8),
    "Toyota Hilux / Pickup":("Toyota Hilux",4.8),
    "Custom (enter below)":("Custom Vehicle",3.5),
}

CSS="""
body,.gradio-container{background:#0a0a14 !important;}
.gradio-container{max-width:1100px !important;margin:0 auto;}
.panel-card{background:#10101e;border:1px solid #1e1e35;border-radius:14px;padding:18px 22px;}
.ds-title{text-align:center;padding:28px 0 20px;font-family:\'Courier New\',monospace;}
.ds-title h1{font-size:2.1rem;font-weight:900;color:#e8e8ff;letter-spacing:.12em;margin:0;}
.ds-title p{color:#555;font-size:.75rem;margin:4px 0 0;letter-spacing:.2em;text-transform:uppercase;}
"""

def update_preset(preset_name):
    name, braking = VEHICLE_PRESETS.get(preset_name,("Custom",3.5))
    return braking, name

with gr.Blocks(css=CSS, title="DriveSense AI") as demo:
    gr.HTML("""<div class=\'ds-title\'><h1>⬡ DRIVESENSE AI</h1><p>Intelligent Dashcam Analysis · Safety Scoring · Driver Feedback</p></div>""")
    with gr.Row():
        with gr.Column(scale=1):
            gr.HTML(\'<div class="panel-card">\')
            gr.Markdown("### 📹 Upload Footage")
            video_input = gr.Video(label="Dashcam / Wearable Camera Video", height=220)
            gr.HTML(\'</div><br>\')
            gr.HTML(\'<div class="panel-card">\')
            gr.Markdown("### ⚙️ Driver & Vehicle Settings")
            vehicle_preset   = gr.Dropdown(choices=list(VEHICLE_PRESETS.keys()), value="Toyota Corolla / Civic", label="Vehicle Preset")
            vehicle_name_box = gr.Textbox(value="Toyota Corolla", label="Vehicle Name", max_lines=1)
            braking_slider   = gr.Slider(minimum=2.0, maximum=8.0, value=3.5, step=0.1, label="0–100 km/h Braking Time (s) — affects safe distance")
            speed_limit_sl   = gr.Slider(minimum=30, maximum=130, value=70, step=5, label="Road Speed Limit (km/h)")
            gr.HTML(\'</div><br>\')
            gr.HTML("""<div class=\'panel-card\' style=\'font-family:monospace;font-size:11px;color:#555;line-height:1.8;\'><b style=\'color:#888;\'>SAFE DISTANCE FORMULA</b><br>d = v×1.5 + v²/(2a)  ×1.2<br><b style=\'color:#888;\'>WEIGHTS</b><br>Following Distance 40%<br>Lane Discipline 35%<br>Traffic Lights 25%</div>""")
            analyse_btn = gr.Button("▶  ANALYSE FOOTAGE", variant="primary")
        with gr.Column(scale=2):
            gr.HTML(\'<div class="panel-card">\')
            gr.Markdown("### 🎬 Annotated Output")
            video_output = gr.Video(label="Processed Video", height=320)
            gr.HTML(\'</div><br>\')
            gr.HTML(\'<div class="panel-card">\')
            gr.Markdown("### 📊 Score Card")
            score_html = gr.HTML()
            gr.HTML(\'</div><br>\')
            gr.HTML(\'<div class="panel-card">\')
            gr.Markdown("### 📋 Full Safety Report")
            report_md = gr.Markdown()
            gr.HTML(\'</div>\')
    vehicle_preset.change(fn=update_preset, inputs=[vehicle_preset], outputs=[braking_slider, vehicle_name_box])
    analyse_btn.click(fn=process_video, inputs=[video_input,vehicle_name_box,braking_slider,speed_limit_sl], outputs=[video_output,report_md,score_html])
    gr.HTML("""<div style=\'text-align:center;padding:24px 0 8px;font-family:monospace;font-size:11px;color:#333;\'>DRIVESENSE AI · EDUCATIONAL USE · ITU LAHORE BSAI</div>""")

if __name__ == "__main__":
    demo.launch(share=True, server_name="0.0.0.0", server_port=7860, show_error=True)
'''
with open('app.py', 'w') as f:
    f.write(app_code.strip())
print('✅ app.py written')

In [ ]:
# ── Cell 6: Verify all files exist ────────────────────────────────────────────
import os
for f in ['app.py', 'detector.py', 'scorer.py']:
    size = os.path.getsize(f)
    print(f'  {f}: {size} bytes ✅' if size > 100 else f'  {f}: ❌ too small!')
print('\nAll files ready.')

In [ ]:
# ── Cell 7: LAUNCH THE APP ────────────────────────────────────────────────────
# This cell starts the server. A public Gradio URL will appear below.
# Click it to open DriveSense AI in a new tab.
!python app.py